# 02 – Données réelles : FiveThirtyEight « nba-elo »

**Source :** fichier `nbaallelo.csv` de FiveThirtyEight, licence **CC BY 4.0**, d'après Basketball-Reference.com.
Analyse juridique : `docs/DATA_SOURCES.md`.

**Objectif :** télécharger le fichier, vérifier qu'il n'a pas changé (empreinte SHA-256), le convertir au schéma
commun, rejouer **exactement** les contrôles du notebook 01, puis examiner chaque alerte.

Aucun modèle de prédiction ici. Le fichier brut n'est pas mis dans Git (voir `.gitignore`) : il est retéléchargé.

## 0. Installation (Colab) ou détection du dépôt (local)

Dans Colab, la cellule clone le dépôt GitHub public, **sans jeton** : aucun secret n'est nécessaire pour lire.

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_USER = "JohnGAB7"
REPO_NAME = "Sport-Analytics"
PROJECT_DIR = "nba"                  # le projet NBA est dans ce sous-dossier du dépôt
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content") / REPO_NAME
    if not repo.exists():
        # Dépôt public : clonage sans jeton (aucun secret dans le notebook ni dans .git/config).
        url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
        subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    repo = repo / PROJECT_DIR
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Exécution locale : on remonte jusqu'au dossier qui contient src/
    here = Path.cwd()
    repo = next(p for p in [here, *here.parents] if (p / "src" / "nba_analytics").exists())
    os.chdir(repo)

sys.path.insert(0, str(repo / "src"))
print("Dossier du dépôt :", Path.cwd().name)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from nba_analytics import quality
from nba_analytics.sources import fivethirtyeight as f38

pd.set_option("display.width", 160)
print(f38.ATTRIBUTION)

## 1. Téléchargement, empreinte et conversion au schéma commun

In [ ]:
raw_path = f38.download("data/raw/nbaallelo.csv")      # s'arrête si l'empreinte a changé
raw = pd.read_csv(raw_path, dtype=str)
print("Fichier brut :", raw.shape, "(deux lignes par match)")
games = f38.to_games(raw)                                # NBA seulement, une ligne par match
print("Table des matchs :", games.shape)
games.head()

## 2. Contrôles de qualité (identiques au notebook 01)

In [ ]:
res = quality.run_all_checks(games)
print("Colonnes obligatoires absentes :", res["missing_columns"] or "aucune")
print("Valeurs non convertibles :", res["conversion_failures"])
print("Types conformes :", bool(res["dtypes"]["ok"].all()))
pd.Series(res["summary"], name="valeur").to_frame()

### 2.1 Scores hors des bornes [50, 200]

Les bornes ont été choisies pour la NBA moderne. On regarde à quelles saisons appartiennent les alertes.

In [ ]:
logic = res["logic"]
print(logic[["same_team", "negative_pts", "implausible_pts", "tie"]].sum())
logic["season"].value_counts().sort_index()

In [ ]:
logic.sort_values("home_pts").head(3)[["game_id", "game_date", "home_team", "away_team", "home_pts", "away_pts"]]

**Lecture :** 51 des 52 alertes datent de 1947 à 1953, avant l'horloge des 24 secondes (1954), quand les scores
étaient bas ; on y trouve le célèbre 19-18 de 1950. La seule alerte moderne est Chicago 49 – Miami 82 (10/04/1999),
un record de faiblesse offensive. Ce sont des **scores réels**, pas des erreurs : on les garde, mais toute
analyse devra traiter les premières saisons à part.

### 2.2 Équipes avec deux matchs le même jour

In [ ]:
cols = ["game_id", "game_date", "home_team", "away_team", "home_pts", "away_pts", "notes"]
d = res["data"]
pairs = set(zip(res["double_booked"]["game_date"], res["double_booked"]["team"]))
mask = [(dt, h) in pairs or (dt, a) in pairs for dt, h, a in zip(d.game_date, d.home_team, d.away_team)]
d[mask][cols]

**Lecture :** sauf en 1954 (deux matchs Milwaukee–Baltimore le même jour, sans note explicative, à vérifier), ces cas sont des **fins de matchs
contestés** rejouées le jour d'un autre match (colonne `notes`). Conséquence pour plus tard : un tel score n'est connu
qu'à la date de reprise ; un backtest ne doit pas l'utiliser avant cette date. Le cas de 2008 vient en plus de la date
probablement erronée signalée en 2.3.

### 2.3 Contrôle propre à la source : date dans `game_id` contre `game_date`

In [ ]:
f38.id_date_mismatches(games)[["game_id", "game_date", "home_team", "away_team", "notes"]]

**Lecture :** deux incohérences internes à la source. Pour le match contesté Atlanta–Miami, la reprise a eu lieu
le 8 mars 2008 (l'identifiant semble juste, la date fausse). Pour la finale 2015 (match 6, Cleveland), le match a eu lieu
le 16 juin 2015 (la date semble juste, l'identifiant faux). Ces deux corrections viennent de faits publics connus et
n'ont **pas** été vérifiées ici contre une seconde source ; les données ne sont donc pas modifiées.

## 3. Statistiques descriptives

In [ ]:
stats = quality.descriptive_stats(d)
stats.loc[[1947, 1955, 1970, 1985, 2000, 2015]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(stats.index, stats["mean_total"], color="#4C72B0")
axes[0].axvline(1955, color="grey", ls=":", lw=1)
axes[0].set(title="Total moyen de points par match", xlabel="saison")
axes[1].plot(stats.index, stats["home_win_rate"], color="#C44E52")
axes[1].axhline(0.5, color="black", lw=0.8)
axes[1].set(title="Part des victoires à domicile", xlabel="saison", ylim=(0.4, 0.8))
plt.tight_layout(); plt.show()

In [ ]:
reg = d[~d["is_playoffs"] & ~d["neutral_site"]]
print("Saison régulière, hors terrain neutre :", len(reg), "matchs")
print("Victoires à domicile :", round((reg.home_pts > reg.away_pts).mean(), 4))
print("Matchs de play-offs :", int(d["is_playoffs"].sum()), "| terrain neutre :", int(d["neutral_site"].sum()))

## 4. Sauvegarde de la table nettoyée (non versionnée)

In [ ]:
out = Path("data/processed/games_fivethirtyeight.csv")
out.parent.mkdir(parents=True, exist_ok=True)
d.to_csv(out, index=False)
print("Écrit :", out, d.shape)

## 5. Conclusion

- 59 008 matchs NBA (1946-47 à 2014-15), aucune valeur manquante, aucun doublon, types conformes.
- Toutes les alertes ont une explication historique ; deux incohérences internes à la source sont signalées, non corrigées.
- La base est utilisable ; les limites sont dans `docs/LIMITATIONS.md`.
- Attribution obligatoire (CC BY 4.0) dans toute publication : voir `f38.ATTRIBUTION`.